In [1]:
from keras.datasets import mnist
import matplotlib.pyplot as plt
import numpy as np
from psc.models.dual_state_frequency_aggregation import PixelStatisticalClassifier
from psc import evaluation

I0000 00:00:1787643322.679987   11936 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787643322.690851   11936 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1787643323.521287   11936 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787643327.432113   11936 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

In [2]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

In [3]:

psc = PixelStatisticalClassifier()
psc.fit(x_train, y_train)
print(psc.classes)
print(psc.statistics.shape)


[0 1 2 3 4 5 6 7 8 9]
(28, 28, 10, 2)


In [4]:
results = evaluation.evaluate(psc, x_test, y_test)
evaluation.display_results(results)

Class         Accuracy   Precision      Recall    F1 Score
----------------------------------------------------------
0               90.44%     100.00%       2.45%       4.78%
1               11.65%      11.38%     100.00%      20.44%
2               89.68%       0.00%       0.00%       0.00%
3               89.90%       0.00%       0.00%       0.00%
4               90.18%       0.00%       0.00%       0.00%
5               91.08%       0.00%       0.00%       0.00%
6               90.42%       0.00%       0.00%       0.00%
7               89.78%     100.00%       0.58%       1.16%
8               90.26%       0.00%       0.00%       0.00%
9               89.91%       0.00%       0.00%       0.00%
overall         11.65%      21.14%      10.30%       2.64%


In [7]:
def display_class_pixel_totals(model):

    # Sum over all pixel positions
    background_totals = model.statistics[:, :, :, 0].sum(axis=(0, 1))
    foreground_totals = model.statistics[:, :, :, 1].sum(axis=(0, 1))

    print(
        f"{'class':<10}"
        f"{'background samples':>24}"
        f"{'foreground samples':>24}"
    )

    print("-" * 58)

    for i, class_label in enumerate(model.classes):

        background = background_totals[i]
        foreground = foreground_totals[i]

        print(
            f"{str(class_label):<10}"
            f"{background:>24,}"
            f"{foreground:>24,}"
        )

display_class_pixel_totals(psc)

class           background samples      foreground samples
----------------------------------------------------------
0                        3,839,054                 804,578
1                        4,884,989                 400,739
2                        3,975,581                 695,491
3                        4,126,073                 680,631
4                        4,026,033                 554,095
5                        3,703,402                 546,662
6                        4,001,405                 638,307
7                        4,350,453                 561,307
8                        3,894,860                 692,324
9                        4,091,665                 572,351


In [6]:
# <<< Analysis >>>
#
# results are shockingly negative. 
# An 11.65% overall accuracy indicates that the model is barely better than a random predictor
# 100% recall paired with having low precision on class 1 indicates, 
# that the model predicted "1" for almost every input image
# 100% precision on class 0 and 7 paired with low recall value indicate that those classes were never predicted.
# Since all classes are almost equally distributed,
# the 11.65% accuracy explains that the model just predicted "1", for almost all cases.
#
# class distributions between binary states shown above tell us why this happens:
# since in the common case, there are way more background pixels than foreground pixels in a training sample
# class frequencies obtained from background pixels greatly outweigh the same from foreground pixels 
# thereby instroducing biasness and skewing predictions

# <<<  Next step >>>
#
# Instead of frequency aggregation, learn probabilities for each class based on state for each pixel
# Then, during prediction, average the probabilities across pixels with the same states
# Finally, use a tunable parameter to choose the weight assigned to the states